# Otimizador Lion

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


## Intuição

Lion ("EvoLved Sign Momentum") foi descoberto por busca. Usa apenas o **sinal** de um momentum interpolado e uma única média móvel exponencial. Mais rápido em TPUs/GPUs (sem second moment por parâmetro) e costuma empatar com AdamW.


## Formulação Matemática

$$c = \beta_1\,m_{t-1} + (1-\beta_1)\,g_t$$
$$\theta_t = \theta_{t-1} - \eta\,(\text{sign}(c) + \lambda \theta_{t-1})$$
$$m_t = \beta_2\,m_{t-1} + (1-\beta_2)\,g_t$$


## Implementação


In [ ]:
import torch
from torch.optim.optimizer import Optimizer


In [ ]:
class Lion(Optimizer):
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        super().__init__(params, dict(lr=lr, betas=betas, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure else None
        for group in self.param_groups:
            lr, (b1, b2), wd = group['lr'], group['betas'], group['weight_decay']
            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad
                state = self.state[p]
                if 'm' not in state:
                    state['m'] = torch.zeros_like(p)
                m = state['m']
                c = b1 * m + (1 - b1) * g
                p.add_(torch.sign(c) + wd * p, alpha=-lr)
                state['m'] = b2 * m + (1 - b2) * g
        return loss


## Experimento


In [ ]:
# Quick sanity: train a 1-layer net to fit y = x with Lion
torch.manual_seed(0)
x = torch.randn(256, 4)
y = x.sum(-1)
net = torch.nn.Linear(4, 1)
opt = Lion(net.parameters(), lr=1e-3, weight_decay=1e-2)
for _ in range(500):
    pred = net(x).squeeze(-1)
    loss = (pred - y).pow(2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
print('final loss:', loss.item())
print('weights ~', net.weight.detach().squeeze().tolist())


## Discussão

- Use um learning rate ~10× menor que o do AdamW (passo de sinal tem magnitude maior).
- Aumente o weight decay (cerca de 5e-2) para compensar.
- Funciona melhor com schedule cosine + warmup, igual ao AdamW.


## Referências

- Repositório da série: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Autor: [Nandobez](https://github.com/Nandobez)
